
# 04_Models_Cheat_Sheet

## What this notebook is for
This is a short demo-day cheat sheet for the **models** part of the project.

It covers:
- `train.py`
- `predict.py`
- `train_metadata.py`
- `predict_metadata.py`
- `train_hybrid.py`
- `predict_hybrid.py`

---

## 1. One-minute model overview

My system has **three stages**:

### Stage 1 — Text-only model
- Uses the **email body text**
- Converts text into **TF-IDF features**
- Trains a **Logistic Regression** classifier

### Stage 2 — Metadata-only model
- Uses **email metadata-style features** extracted from the email text/header content
- Trains a **Logistic Regression** classifier on those metadata features

### Stage 3 — Hybrid stacking model
- Combines:
  - Stage 1 text probability
  - Stage 2 metadata probability
- Feeds both into a final **meta Logistic Regression** model

### Simple demo line
> “The project uses a three-stage design: a text-only model, a metadata-only model, and then a hybrid stacking model that combines both probability signals into one final decision.”

---



# 2. Why the three models matter

### Stage 1
Good at learning suspicious wording, phrases, and linguistic patterns.

### Stage 2
Good at using structural or metadata-style clues that may still be useful even if the wording is changed.

### Stage 3
Combines both views of the email so the system is less dependent on only one signal type.

### Good demo line
> “The key idea is that phishing can be detected from both language patterns and metadata-related cues, so the hybrid model tries to benefit from both rather than relying on only one source.”

---



# 3. File-by-file cheat sheet

---

## `train.py` — Stage 1 training

### What it does
This trains the **text-only phishing model**.

### How it works
1. loads the cleaned email dataset  
2. splits it into train and test  
3. turns the text into **TF-IDF features**  
4. trains a **Logistic Regression** model  
5. evaluates it using accuracy, precision, recall, F1, ROC-AUC, PR-AUC, confusion matrix, and classification report  
6. finds the best threshold by F1  
7. saves the model, vectorizer, results JSON, and test prediction CSVs

### What the model is actually learning
It learns which words or word combinations are associated with phishing vs legitimate email.

### What to say in the demo
> “Stage 1 is my text-only baseline. It uses TF-IDF to convert email text into numeric features, then Logistic Regression learns which language patterns are associated with phishing.”

### Short speech topics
- text baseline
- TF-IDF representation
- probabilistic classifier
- saved per-sample predictions for evaluation

---

## `predict.py` — Stage 1 inference

### What it does
This loads the saved Stage 1 model and predicts on a new email.

### How it works
1. loads `model.joblib` and `vectorizer.joblib`  
2. transforms the input email into TF-IDF features  
3. gets:
   - class prediction
   - phishing probability  
4. returns a simple decision:
   - phishing
   - legit
   - uncertain

### Important detail
It uses an **uncertainty band**:
- if probability is between `0.30` and `0.70`, it labels the result as **uncertain**

### What to say in the demo
> “`predict.py` is the inference script for Stage 1. It takes a new email, vectorises the text, gets a phishing probability, and then converts that into a readable decision.”

---

## `train_metadata.py` — Stage 2 training

### What it does
This trains the **metadata-only model**.

### How it works
1. loads the dataset  
2. splits train and test  
3. extracts metadata-style features  
4. trains a **Logistic Regression** classifier  
5. evaluates it with standard metrics  
6. saves:
   - model
   - vectorizer
   - results JSON
   - test prediction CSV
   - error analysis CSV
   - top metadata features CSV

### Extra useful part
This script also runs a **label-shuffle sanity check**.  
That checks whether performance collapses when labels are randomly shuffled, which helps show the model is learning something real rather than noise.

### What the model is actually learning
It learns whether metadata-related patterns are associated with phishing, rather than relying on the wording of the email body.

### What to say in the demo
> “Stage 2 isolates metadata-style signals. That lets me test whether header and structural clues can still detect phishing independently of normal wording.”

### Short speech topics
- metadata-only baseline
- interpretability through top features
- sanity checking with label shuffle
- error analysis export

---

## `predict_metadata.py` — Stage 2 inference

### What it does
This loads the Stage 2 model and predicts on a new email using metadata features only.

### How it works
1. loads the saved metadata model and vectorizer  
2. transforms the email into metadata features  
3. outputs:
   - class prediction
   - phishing probability
   - decision label

### Important detail
It uses **three decision zones**:
- `>= 0.70` → Phishing  
- `<= 0.30` → Legit  
- between them → Uncertain

### What to say in the demo
> “This is the Stage 2 prediction script. It scores the email using metadata-related features, then turns the probability into phishing, legit, or uncertain.”

---

## `train_hybrid.py` — Stage 3 hybrid training

### What it does
This trains the **hybrid stacking model**, which is the main final model in the project.

### How it works
1. loads the dataset  
2. reuses the same saved train/test split for fair comparison across stages  
3. uses **Stratified K-Fold** on the training split  
4. for each fold:
   - train a text model
   - train a metadata model
   - generate out-of-fold probabilities  
5. stack those probabilities into a 2-column training matrix  
6. train a **meta Logistic Regression** model on top  
7. train final Stage 1 and Stage 2 base models on the full training split  
8. generate final test probabilities:
   - text probability
   - metadata probability
   - hybrid probability  
9. evaluate and save artifacts, results, error analysis, and test prediction CSVs

### Why this is stronger than just averaging
The final meta-model learns **how much to trust** the text and metadata probabilities, rather than treating them as equally important.

### What to say in the demo
> “Stage 3 is a stacking model. Instead of choosing between text and metadata, it learns from both. The final model takes the text-model probability and the metadata-model probability, then combines them into one hybrid prediction.”

### Short speech topics
- stacking
- out-of-fold probabilities
- fair split reuse
- meta-model combines both signals

### Easy explanation if asked “what is stacking?”
> “Stacking means one model learns from the outputs of other models.”

---

## `predict_hybrid.py` — Stage 3 inference

### What it does
This is the inference script for the final hybrid model.

### How it works
1. loads:
   - text model
   - text vectorizer
   - metadata model
   - metadata vectorizer
   - meta model  
2. gets a text probability from the text model  
3. gets a metadata probability from the metadata model  
4. sends both into the meta model  
5. outputs:
   - final hybrid prediction
   - final hybrid probability
   - text probability
   - metadata probability
   - readable decision

### Important detail
It also uses an **uncertainty band**:
- between `0.30` and `0.70` → uncertain

### Why this script is useful in a demo
It lets you explain the hybrid decision transparently:
- “text model says X”
- “metadata model says Y”
- “final hybrid model combines both”

### What to say in the demo
> “`predict_hybrid.py` is useful because it shows the whole decision chain. It does not just give the final result; it also shows the text probability and metadata probability that were combined into the final hybrid score.”

---



# 4. Quick comparison table

| Stage | Main idea | Input | Model type | Main strength |
|---|---|---|---|---|
| Stage 1 | Text-only | Email text | Logistic Regression on TF-IDF | Learns suspicious wording patterns |
| Stage 2 | Metadata-only | Metadata-style features | Logistic Regression | Uses structural / metadata cues |
| Stage 3 | Hybrid stacking | Stage 1 prob + Stage 2 prob | Meta Logistic Regression | Combines both signal sources |

---



# 5. Very short speech notes

## Stage 1
> “Stage 1 is the text-only baseline. It uses TF-IDF and Logistic Regression to learn phishing-related language patterns.”

## Stage 2
> “Stage 2 is the metadata-only model. It checks whether structural and metadata-related clues can detect phishing independently of the email wording.”

## Stage 3
> “Stage 3 is the final hybrid stacking model. It takes the probabilities from the text model and metadata model, then learns how to combine them into one final prediction.”

---



# 6. Likely demo questions

## “Why use Logistic Regression?”
> “It gives a strong and simple probabilistic baseline, works well with sparse features, and is easy to evaluate and compare across stages.”

## “Why use TF-IDF in Stage 1?”
> “TF-IDF turns text into numeric features based on term importance, so the classifier can learn which wording patterns are associated with phishing.”

## “Why have a metadata-only model at all?”
> “Because phishing can sometimes evade text-based detection, so I wanted to test whether metadata-related cues still provide useful signal.”

## “Why is Stage 3 better?”
> “Because it combines two different views of the same email rather than depending on only one source of evidence.”

## “What does the hybrid model actually combine?”
> “It combines the Stage 1 text probability and the Stage 2 metadata probability using a final meta-model.”

---



# 7. Best closing line

> “The models folder shows the progression from a single-source baseline to a multi-source hybrid system: first text-only, then metadata-only, and finally a stacking model that combines both for a stronger final prediction.”
